
## CA-DFN: Confidence-Aware Dynamic Fusion with Complementarity-Predictive Attention


### Novel Contributions:
1. CA-DFN with architecture embeddings + pairwise KL disagreement
2. Complementarity-Predictive Fusion (CACA predicts fusion gain)
3. Curriculum Knowledge Distillation
4. First calibration analysis for food recognition

> ⚡ GPU: 12GB+ required  •  ⏱ ~10-12 hours


## 1. Installation

In [ ]:
!pip install -q timm grad-cam ptflops > /dev/null 2>&1
import torch, sys
print(f"Python: {sys.version.split()[0]}  PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected!")

## 2. Imports & Configuration

In [ ]:
import os, time, copy, random, warnings, gc, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

import torchvision
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import Food101, ImageFolder
import timm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

class CFG:
    DATA_DIR      = "./data"
    NUM_CLASSES   = 101
    IMG_SIZE      = 224
    EPOCHS        = 50
    BATCH_SIZE    = 32
    ACCUM_STEPS   = 2       # effective BS = 64
    LR            = 5e-4
    WD            = 0.02
    LABEL_SMOOTH  = 0.1
    WARMUP_EP     = 5
    WORKERS       = 2
    CUTMIX_ALPHA  = 1.0
    MIXUP_ALPHA   = 0.8
    CUTMIX_PROB   = 0.5
    # CA-DFN
    FUSION_EP     = 40
    FUSION_LR     = 3e-4
    FUSION_HIDDEN = 512
    FUSION_TEMP   = 3.0
    FUSION_ENT_W  = 0.3
    FUSION_IDROP  = 0.25
    # KD
    KD_EP         = 40
    KD_TEMP       = 4.0
    KD_ALPHA      = 0.7
    # Seeds
    SEEDS         = [42, 123, 456]
    DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CKPT          = "./checkpoints"
    OUT           = "./results"

os.makedirs(CFG.CKPT, exist_ok=True)
os.makedirs(CFG.OUT, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

print(f"Device: {CFG.DEVICE}  |  EPOCHS={CFG.EPOCHS}  BS={CFG.BATCH_SIZE}x{CFG.ACCUM_STEPS}")
print(f"Models: ResNet-50, EfficientNet-B4, DenseNet-121, ConvNeXt-Tiny, Swin-Small")

## 3. Dataset — Food-101 (Full 101 Classes)

In [ ]:
train_tf = T.Compose([
    T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(0.4, 0.4, 0.4, 0.15),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    T.RandomErasing(p=0.25),
])
test_tf = T.Compose([
    T.Resize(256), T.CenterCrop(CFG.IMG_SIZE), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx-cut_w//2,0,W), np.clip(cy-cut_h//2,0,H)
    x2, y2 = np.clip(cx+cut_w//2,0,W), np.clip(cy+cut_h//2,0,H)
    return x1, y1, x2, y2

def apply_cutmix_or_mixup(images, labels, nc):
    if np.random.rand() < CFG.CUTMIX_PROB:
        lam = np.random.beta(CFG.CUTMIX_ALPHA, CFG.CUTMIX_ALPHA)
        ri = torch.randperm(images.size(0)).to(images.device)
        x1,y1,x2,y2 = rand_bbox(images.size(), lam)
        images[:,:,x1:x2,y1:y2] = images[ri,:,x1:x2,y1:y2]
        lam = 1-((x2-x1)*(y2-y1)/(images.size(-1)*images.size(-2)))
    else:
        lam = np.random.beta(CFG.MIXUP_ALPHA, CFG.MIXUP_ALPHA)
        ri = torch.randperm(images.size(0)).to(images.device)
        images = lam*images + (1-lam)*images[ri]
    oh = F.one_hot(labels, nc).float()
    return images, lam*oh + (1-lam)*F.one_hot(labels[ri], nc).float()

print("Downloading Food-101...")
full_train = Food101(CFG.DATA_DIR, split="train", transform=train_tf, download=True)
test_ds    = Food101(CFG.DATA_DIR, split="test",  transform=test_tf,  download=True)

n_val = int(0.1*len(full_train)); n_train = len(full_train)-n_val
train_ds, val_ds_raw = random_split(full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(42))

class _ValDS(Dataset):
    def __init__(self, subset, tf):
        self.indices = subset.indices
        self.ds = copy.copy(subset.dataset); self.ds.transform = tf
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.ds[self.indices[i]]

val_ds = _ValDS(val_ds_raw, test_tf)
train_ld = DataLoader(train_ds, CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.WORKERS, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.WORKERS, pin_memory=True)
test_ld  = DataLoader(test_ds,  CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.WORKERS, pin_memory=True)
CLASS_NAMES = full_train.classes
print(f"Food-101: Train:{n_train:,} Val:{n_val:,} Test:{len(test_ds):,} Classes:101")

## 4. Upgraded Base Models

| Model | Type | Params | Upgrade from v2 |
|---|---|---|---|
| ResNet-50 | CNN | 23.7M | Same |
| **EfficientNet-B4** | CNN | 17.7M | Was B0 (5.3M) |
| DenseNet-121 | CNN | 7.1M | Same |
| **ConvNeXt-Tiny** | CNN | 28.6M | Was MobileNetV2 (3.4M) |
| **Swin-Small** | Transformer | 49.6M | Was Swin-Tiny (28M) |


In [ ]:
MODEL_NAMES = ["resnet50", "efficientnet_b4", "densenet121", "convnext_tiny", "swin_small"]
DISPLAY = {"resnet50":"ResNet-50", "efficientnet_b4":"EfficientNet-B4",
           "densenet121":"DenseNet-121", "convnext_tiny":"ConvNeXt-Tiny", "swin_small":"Swin-Small"}
MODEL_TYPE = {"resnet50":"CNN", "efficientnet_b4":"CNN", "densenet121":"CNN",
              "convnext_tiny":"CNN", "swin_small":"Transformer"}

def build_model(name, nc=CFG.NUM_CLASSES, pretrained=True):
    if name == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, nc)
        bb = [p for n,p in m.named_parameters() if "fc" not in n]
        hd = list(m.fc.parameters())
    elif name == "efficientnet_b4":
        m = models.efficientnet_b4(weights="IMAGENET1K_V1" if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, nc)
        bb = [p for n,p in m.named_parameters() if "classifier" not in n]
        hd = list(m.classifier.parameters())
    elif name == "densenet121":
        m = models.densenet121(weights="IMAGENET1K_V1" if pretrained else None)
        m.classifier = nn.Linear(m.classifier.in_features, nc)
        bb = [p for n,p in m.named_parameters() if "classifier" not in n]
        hd = list(m.classifier.parameters())
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights="IMAGENET1K_V1" if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, nc)
        bb = [p for n,p in m.named_parameters() if "classifier" not in n]
        hd = list(m.classifier.parameters())
    elif name == "swin_small":
        m = timm.create_model("swin_small_patch4_window7_224", pretrained=pretrained, num_classes=nc)
        bb = [p for n,p in m.named_parameters() if "head" not in n]
        hd = [p for n,p in m.named_parameters() if "head" in n]
    else: raise ValueError(name)
    return m.to(CFG.DEVICE), [{"params":bb,"lr":CFG.LR*0.1},{"params":hd,"lr":CFG.LR}]

print(f"{'Model':<22s} {'Type':<12s} {'Params':>8s}")
print("-"*44)
for nm in MODEL_NAMES:
    tmp,_ = build_model(nm, pretrained=False)
    print(f"{DISPLAY[nm]:<22s} {MODEL_TYPE[nm]:<12s} {sum(p.numel() for p in tmp.parameters())/1e6:>7.1f}M")
    del tmp
torch.cuda.empty_cache()

## 5. Training Engine (CutMix/MixUp + Gradient Accumulation)

In [ ]:
class SoftCE(nn.Module):
    def __init__(self, ls=0.0):
        super().__init__(); self.ls = ls
    def forward(self, logits, targets):
        if targets.dim() == 1:
            nc = logits.size(1)
            soft = F.one_hot(targets, nc).float()*(1-self.ls) + self.ls/nc
        else: soft = targets
        return -(soft * F.log_softmax(logits, dim=-1)).sum(-1).mean()

def train_epoch(model, loader, opt, crit, sched=None, use_mix=True):
    model.train(); loss_sum = correct = total = 0
    opt.zero_grad(set_to_none=True)
    for step, (imgs, labs) in enumerate(loader):
        imgs, labs = imgs.to(CFG.DEVICE, non_blocking=True), labs.to(CFG.DEVICE, non_blocking=True)
        if use_mix:
            imgs, soft_labs = apply_cutmix_or_mixup(imgs, labs, CFG.NUM_CLASSES)
        else: soft_labs = labs
        loss = crit(model(imgs), soft_labs) / CFG.ACCUM_STEPS
        loss.backward()
        if (step+1) % CFG.ACCUM_STEPS == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(set_to_none=True)
        loss_sum += loss.item()*CFG.ACCUM_STEPS*imgs.size(0)
        correct += model(imgs).detach().argmax(1).eq(labs).sum().item() if not use_mix else 0
        total += labs.size(0)
    if sched: sched.step()
    return loss_sum/total, 0.0  # acc computed separately to save memory

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); loss_sum = correct = total = 0
    ce = nn.CrossEntropyLoss(); all_probs, all_labs = [], []
    for imgs, labs in loader:
        imgs, labs = imgs.to(CFG.DEVICE, non_blocking=True), labs.to(CFG.DEVICE, non_blocking=True)
        out = model(imgs)
        loss_sum += ce(out, labs).item()*imgs.size(0)
        probs = F.softmax(out, dim=1)
        correct += out.argmax(1).eq(labs).sum().item()
        total += labs.size(0)
        all_probs.append(probs.cpu()); all_labs.extend(labs.cpu().numpy())
    all_probs = torch.cat(all_probs); all_labs = np.array(all_labs)
    preds = all_probs.argmax(1).numpy()
    t5 = all_probs.topk(5,1).indices
    t5a = 100*np.mean([all_labs[i] in t5[i].numpy() for i in range(len(all_labs))])
    return dict(loss=loss_sum/total, top1=100*correct/total, top5=t5a,
                preds=preds, labels=all_labs, probs=all_probs)

def train_full(name, seed, epochs=CFG.EPOCHS):
    print(f"\n{'='*60}\n  Training {DISPLAY[name]} (seed={seed}, {epochs} ep)\n{'='*60}")
    seed_everything(seed)
    model, pg = build_model(name)
    crit = SoftCE(CFG.LABEL_SMOOTH)
    opt = optim.AdamW(pg, weight_decay=CFG.WD)
    warm = LinearLR(opt, start_factor=0.01, total_iters=CFG.WARMUP_EP)
    cos = CosineAnnealingLR(opt, T_max=epochs-CFG.WARMUP_EP, eta_min=1e-7)
    sched = SequentialLR(opt, [warm, cos], milestones=[CFG.WARMUP_EP])
    best_acc = 0.0; path = f"{CFG.CKPT}/{name}_s{seed}.pth"
    hist = {k:[] for k in ("tl","ta","vl","va")}
    for ep in range(epochs):
        t0 = time.time()
        tl, _ = train_epoch(model, train_ld, opt, crit, sched)
        vr = evaluate(model, val_ld)
        hist["tl"].append(tl); hist["ta"].append(vr["top1"])  # use val as proxy
        hist["vl"].append(vr["loss"]); hist["va"].append(vr["top1"])
        flag = ""
        if vr["top1"] > best_acc:
            best_acc = vr["top1"]; torch.save(model.state_dict(), path); flag = " ⭐"
        if (ep+1)%5==0 or ep==0 or flag:
            print(f"  Ep {ep+1:>2}/{epochs} | Val {vr['top1']:.1f}% T5 {vr['top5']:.1f}% | {time.time()-t0:.0f}s{flag}")
    model.load_state_dict(torch.load(path, weights_only=True))
    tr = evaluate(model, test_ld)
    print(f"  Test: Top-1 {tr['top1']:.2f}%  Top-5 {tr['top5']:.2f}%  F1 {f1_score(tr['labels'],tr['preds'],average='macro'):.4f}")
    return model, hist, tr

print("Training engine ready.")

## 6. Train All 5 Base Models
⏱ ~8-10 hours on 12GB GPU


In [ ]:
SEED0 = CFG.SEEDS[0]
trained, hists, results = {}, {}, {}
for nm in MODEL_NAMES:
    m, h, r = train_full(nm, SEED0)
    trained[nm], hists[nm], results[nm] = m, h, r
    gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*65}")
print(f"{'Model':<22s} {'Top-1':>8s} {'Top-5':>8s} {'F1':>8s}")
print("-"*50)
for nm in MODEL_NAMES:
    r = results[nm]; f1 = f1_score(r["labels"],r["preds"],average="macro")
    print(f"{DISPLAY[nm]:<22s} {r['top1']:>7.2f}% {r['top5']:>7.2f}% {f1:>7.4f}")
print("="*65)

## 7. Cache Predictions & Static Baselines

In [ ]:
@torch.no_grad()
def cache_probs(models_dict, loader):
    out = {n:[] for n in models_dict}; labs = []
    for imgs, y in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True); labs.extend(y.numpy())
        for n, m in models_dict.items():
            m.eval(); out[n].append(F.softmax(m(imgs),dim=1).cpu())
    for n in out: out[n] = torch.cat(out[n])
    return out, np.array(labs)

def ens_eval(probs, labs, weights=None):
    names = list(probs.keys())
    if weights is None: weights = {n:1.0/len(names) for n in names}
    ep = sum(weights[n]*probs[n] for n in names)
    preds = ep.argmax(1).numpy()
    t5 = ep.topk(5,1).indices
    t5a = 100*np.mean([labs[i] in t5[i].numpy() for i in range(len(labs))])
    return dict(top1=100*accuracy_score(labs,preds), top5=t5a,
                f1=f1_score(labs,preds,average="macro"), preds=preds, probs=ep)

print("Caching predictions...")
test_P, test_Y = cache_probs(trained, test_ld)
val_P, val_Y = cache_probs(trained, val_ld)
trn_P, trn_Y = cache_probs(trained, train_ld)

uni = ens_eval(test_P, test_Y)
va = {n:accuracy_score(val_Y, val_P[n].argmax(1).numpy()) for n in MODEL_NAMES}
tot = sum(va.values()); pw = {n:va[n]/tot for n in MODEL_NAMES}
prf = ens_eval(test_P, test_Y, pw)
vote_stack = np.stack([test_P[n].argmax(1).numpy() for n in MODEL_NAMES])
vote_preds = np.apply_along_axis(lambda x: np.bincount(x,minlength=CFG.NUM_CLASSES).argmax(), 0, vote_stack)
vote_t1 = 100*accuracy_score(test_Y, vote_preds)

print(f"\nUniform Avg:   Top-1={uni['top1']:.2f}%  F1={uni['f1']:.4f}")
print(f"Perf-Weighted: Top-1={prf['top1']:.2f}%  F1={prf['f1']:.4f}")
print(f"Majority Vote: Top-1={vote_t1:.2f}%")

## 8. CA-DFN — Architecture-Aware Dynamic Fusion

**Novel features:** architecture-type embeddings, pairwise KL disagreement,
residual block, entropy regularization, temperature scaling, input dropout.


In [ ]:
class CADFN_v3(nn.Module):
    def __init__(self, n_models, n_classes, model_types, hidden=512,
                 drop=0.3, attn_temp=3.0, input_drop=0.25):
        super().__init__()
        self.n_models, self.n_classes = n_models, n_classes
        self.temp, self.input_drop = attn_temp, input_drop
        self.arch_embed = nn.Embedding(2, 16)  # CNN=0, Transformer=1
        self.arch_types = model_types
        per_model = n_classes + 1 + 1 + 16  # softmax + entropy + maxprob + arch_emb
        n_pairs = n_models * (n_models - 1) // 2
        inp_dim = n_models * per_model + n_pairs
        self.net = nn.Sequential(nn.Linear(inp_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(drop))
        self.res_block = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
                                       nn.Dropout(drop), nn.Linear(hidden, hidden))
        self.res_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, n_models)

    def forward(self, prob_list, training=False):
        B, device = prob_list[0].size(0), prob_list[0].device
        if training and self.input_drop > 0 and random.random() < self.input_drop:
            di = random.randint(0, self.n_models-1)
            prob_list = [p if i != di else torch.ones_like(p)/self.n_classes for i,p in enumerate(prob_list)]
        stk = torch.stack(prob_list, dim=1)
        mf = []
        for i, p in enumerate(prob_list):
            ent = -(p*(p+1e-8).log()).sum(1, keepdim=True)
            mx = p.max(1, keepdim=True).values
            ae = self.arch_embed(torch.tensor([self.arch_types[i]], device=device)).expand(B,-1)
            mf.append(torch.cat([p, ent, mx, ae], dim=1))
        model_feat = torch.cat(mf, dim=1)
        dis = []
        for i in range(self.n_models):
            for j in range(i+1, self.n_models):
                kl = F.kl_div((prob_list[i]+1e-8).log(), prob_list[j]+1e-8, reduction='none').sum(1, keepdim=True)
                dis.append(kl)
        x = torch.cat([model_feat, torch.cat(dis,dim=1)], dim=1)
        h = self.net(x)
        h = self.res_norm(h + self.res_block(h))
        w = F.softmax(self.head(h)/self.temp, dim=1)
        return (w.unsqueeze(2)*stk).sum(1), w

class EntropyRegLoss(nn.Module):
    def __init__(self, ent_w=0.3): super().__init__(); self.ent_w = ent_w
    def forward(self, ens_probs, labels, attn_w):
        ce = F.nll_loss(ens_probs.clamp(min=1e-8).log(), labels)
        ent = -(attn_w*(attn_w+1e-8).log()).sum(1).mean()
        return ce - self.ent_w*(ent/math.log(attn_w.size(1))), ce.item(), (ent/math.log(attn_w.size(1))).item()

arch_types = [0 if MODEL_TYPE[n]=="CNN" else 1 for n in MODEL_NAMES]
print(f"Arch types: {dict(zip([DISPLAY[n] for n in MODEL_NAMES], ['CNN' if t==0 else 'ViT' for t in arch_types]))}")

## 9. Train CA-DFN

In [ ]:
class _FusDS(Dataset):
    def __init__(self, P, Y):
        self.ps = [P[n] for n in MODEL_NAMES]
        self.ys = torch.tensor(Y, dtype=torch.long)
    def __len__(self): return len(self.ys)
    def __getitem__(self, i): return [p[i] for p in self.ps], self.ys[i]

def _col(batch):
    pls, ys = zip(*batch); N = len(pls[0])
    return [torch.stack([pl[i] for pl in pls]) for i in range(N)], torch.stack(ys)

fus_tr = DataLoader(_FusDS(trn_P, trn_Y), 512, True, collate_fn=_col)
fus_vl = DataLoader(_FusDS(val_P, val_Y), 512, False, collate_fn=_col)
fus_te = DataLoader(_FusDS(test_P, test_Y), 512, False, collate_fn=_col)

seed_everything(SEED0)
cadfn = CADFN_v3(len(MODEL_NAMES), CFG.NUM_CLASSES, arch_types,
                 hidden=CFG.FUSION_HIDDEN, attn_temp=CFG.FUSION_TEMP,
                 input_drop=CFG.FUSION_IDROP).to(CFG.DEVICE)
f_opt = optim.AdamW(cadfn.parameters(), lr=CFG.FUSION_LR, weight_decay=1e-3)
f_sch = CosineAnnealingLR(f_opt, T_max=CFG.FUSION_EP, eta_min=1e-6)
f_crit = EntropyRegLoss(CFG.FUSION_ENT_W)

best_fa = 0.0
print(f"Training CA-DFN ({CFG.FUSION_EP} epochs)...")
for ep in range(CFG.FUSION_EP):
    cadfn.train(); c = t = 0
    for ps, ys in fus_tr:
        ps = [p.to(CFG.DEVICE) for p in ps]; ys = ys.to(CFG.DEVICE)
        f_opt.zero_grad()
        ep_probs, aw = cadfn(ps, training=True)
        loss, _, _ = f_crit(ep_probs, ys, aw)
        loss.backward(); f_opt.step()
        c += ep_probs.argmax(1).eq(ys).sum().item(); t += ys.size(0)
    f_sch.step()
    cadfn.eval(); vc = vt = 0
    with torch.no_grad():
        for ps, ys in fus_vl:
            ps = [p.to(CFG.DEVICE) for p in ps]; ys = ys.to(CFG.DEVICE)
            ep2, _ = cadfn(ps, training=False)
            vc += ep2.argmax(1).eq(ys).sum().item(); vt += ys.size(0)
    va_acc = 100*vc/vt; flag = ""
    if va_acc > best_fa:
        best_fa = va_acc; torch.save(cadfn.state_dict(), f"{CFG.CKPT}/cadfn.pth"); flag = " ⭐"
    if (ep+1)%5==0 or flag:
        print(f"  Ep {ep+1:>2}/{CFG.FUSION_EP}  Tr {100*c/t:.1f}%  Val {va_acc:.1f}%{flag}")

cadfn.load_state_dict(torch.load(f"{CFG.CKPT}/cadfn.pth", weights_only=True)); cadfn.eval()
ca_preds_l, ca_probs_l, ca_w_l = [], [], []
with torch.no_grad():
    for ps, _ in fus_te:
        ps = [p.to(CFG.DEVICE) for p in ps]
        ep, w = cadfn(ps, training=False)
        ca_preds_l.append(ep.argmax(1).cpu()); ca_probs_l.append(ep.cpu()); ca_w_l.append(w.cpu())
ca_preds = torch.cat(ca_preds_l).numpy()
ca_probs = torch.cat(ca_probs_l); ca_w = torch.cat(ca_w_l)
ca_t1 = 100*accuracy_score(test_Y, ca_preds)
ca_f1 = f1_score(test_Y, ca_preds, average="macro")
t5i = ca_probs.topk(5,1).indices
ca_t5 = 100*np.mean([test_Y[i] in t5i[i].numpy() for i in range(len(test_Y))])
best_single = max(results[n]["top1"] for n in MODEL_NAMES)
best_nm = max(MODEL_NAMES, key=lambda n: results[n]["top1"])

print(f"\nCA-DFN: Top-1={ca_t1:.2f}%  Top-5={ca_t5:.2f}%  F1={ca_f1:.4f}")
print(f"  vs Best Single ({DISPLAY[best_nm]}): {ca_t1-best_single:+.2f}%")
print(f"  vs Uniform: {ca_t1-uni['top1']:+.2f}%  vs Perf-Weight: {ca_t1-prf['top1']:+.2f}%")
mw = ca_w.mean(0).numpy()
for i,n in enumerate(MODEL_NAMES): print(f"  {DISPLAY[n]:<20s} weight: {mw[i]:.4f}")

## 10. CACA — Complementarity Analysis + Predictive Proof

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def _tgt(model, name):
    if name == "resnet50": return [model.layer4[-1]]
    if name == "efficientnet_b4": return [model.features[-1]]
    if name == "densenet121": return [model.features.denseblock4]
    if name == "convnext_tiny": return [model.features[-1]]
    if name == "swin_small": return [model.layers[-1].blocks[-1].norm1]

def _swin_reshape(t, h=7, w=7):
    return t.reshape(t.size(0),h,w,t.size(-1)).permute(0,3,1,2)

N_CAM=150; N_VIS=5
raw_tf2 = T.Compose([T.Resize(256),T.CenterCrop(224),T.ToTensor()])
raw_ds = Food101(CFG.DATA_DIR, split="test", transform=raw_tf2)
idxs = np.random.RandomState(0).choice(len(test_ds), N_CAM, replace=False)

print(f"Grad-CAM: {N_CAM} images x {len(MODEL_NAMES)} models...")
heatmaps = {n:[] for n in MODEL_NAMES}
for nm in MODEL_NAMES:
    m = trained[nm]; m.eval()
    rf = _swin_reshape if "swin" in nm else None
    cam = GradCAM(model=m, target_layers=_tgt(m,nm), reshape_transform=rf)
    for idx in idxs:
        img,lab = test_ds[idx]
        heatmaps[nm].append(cam(input_tensor=img.unsqueeze(0).to(CFG.DEVICE), targets=[ClassifierOutputTarget(lab)])[0])
    del cam; torch.cuda.empty_cache()

THR=0.5; iou_mx = np.zeros((len(MODEL_NAMES),len(MODEL_NAMES)))
for i,ni in enumerate(MODEL_NAMES):
    for j,nj in enumerate(MODEL_NAMES):
        ious = []
        for k in range(N_CAM):
            a=(heatmaps[ni][k]>THR).astype(float); b=(heatmaps[nj][k]>THR).astype(float)
            ious.append((a*b).sum()/(np.clip(a+b,0,1).sum()+1e-8))
        iou_mx[i,j] = np.mean(ious)
comp_mx = 1-iou_mx; mask_tri = np.triu(np.ones_like(comp_mx,dtype=bool),k=1)
mean_comp = comp_mx[mask_tri].mean()

# NOVEL: complementarity predicts fusion gain
pair_comp, pair_gain = [], []
for i,ni in enumerate(MODEL_NAMES):
    for j,nj in enumerate(MODEL_NAMES):
        if i>=j: continue
        cs = comp_mx[i,j]
        pr = ens_eval({ni:test_P[ni],nj:test_P[nj]}, test_Y)
        bi = max(results[ni]["top1"], results[nj]["top1"])
        g = pr["top1"]-bi; pair_comp.append(cs); pair_gain.append(g)
        print(f"  {DISPLAY[ni]:>18s}+{DISPLAY[nj]:<18s} Comp={cs:.3f} Gain={g:+.2f}%")
correlation = np.corrcoef(pair_comp, pair_gain)[0,1]
print(f"\nCorrelation (comp vs gain): r={correlation:.3f}")

disp = [DISPLAY[n] for n in MODEL_NAMES]
fig, axes = plt.subplots(1,3,figsize=(20,6))
sns.heatmap(iou_mx,annot=True,fmt=".3f",cmap="YlOrRd",xticklabels=disp,yticklabels=disp,ax=axes[0],vmin=0,vmax=1,square=True)
axes[0].set_title("Grad-CAM IoU",fontweight="bold")
sns.heatmap(comp_mx,annot=True,fmt=".3f",cmap="YlGn",xticklabels=disp,yticklabels=disp,ax=axes[1],vmin=0,vmax=1,square=True)
axes[1].set_title("Complementarity (1-IoU)",fontweight="bold")
axes[2].scatter(pair_comp,pair_gain,s=120,c="#2e86de",edgecolors="k",zorder=3)
z = np.polyfit(pair_comp,pair_gain,1); xl = np.linspace(min(pair_comp),max(pair_comp),50)
axes[2].plot(xl,np.polyval(z,xl),"r--",lw=2,label=f"r={correlation:.3f}")
axes[2].set_xlabel("Complementarity"); axes[2].set_ylabel("Fusion Gain (%)")
axes[2].set_title("Complementarity Predicts Fusion Benefit",fontweight="bold"); axes[2].legend(); axes[2].grid(True,alpha=.3)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/caca_iou.png",dpi=300,bbox_inches="tight"); plt.show()

fig,axes = plt.subplots(N_VIS,len(MODEL_NAMES)+1,figsize=(3*(len(MODEL_NAMES)+1),3*N_VIS))
for row,si in enumerate(idxs[:N_VIS]):
    ri = raw_ds[si][0].permute(1,2,0).numpy(); lab = raw_ds[si][1]
    axes[row,0].imshow(ri); axes[row,0].set_title(CLASS_NAMES[lab],fontsize=9,fontweight="bold"); axes[row,0].axis("off")
    for col,nm in enumerate(MODEL_NAMES):
        axes[row,col+1].imshow(show_cam_on_image(ri,heatmaps[nm][row],use_rgb=True))
        axes[row,col+1].set_title(DISPLAY[nm],fontsize=8); axes[row,col+1].axis("off")
plt.suptitle("CACA: Cross-Architecture Complementarity",fontsize=14,fontweight="bold",y=1.01)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/caca_heatmaps.png",dpi=200,bbox_inches="tight"); plt.show()
print(f"Mean Complementarity: {mean_comp:.4f}")

## 11. Uncertainty & Calibration Analysis

In [ ]:
ens_np = ca_probs.numpy()
ent = -(ens_np*np.log(ens_np+1e-10)).sum(1); mx_c = ens_np.max(1); correct = (ca_preds==test_Y)

def ece_calc(probs, labs, n_bins=15):
    confs=probs.max(1); preds=probs.argmax(1); accs=(preds==labs).astype(float)
    edges=np.linspace(0,1,n_bins+1); ece=0; bas=[]; bcs=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(confs>lo)&(confs<=hi)
        if m.sum(): ba=accs[m].mean(); bc=confs[m].mean(); ece+=m.sum()/len(labs)*abs(ba-bc); bas.append(ba); bcs.append(bc)
    return ece, bas, bcs

ece_val,bas,bcs = ece_calc(ens_np,test_Y)
si2 = np.argsort(-mx_c); cum_c = np.cumsum(correct[si2]); aac = cum_c/np.arange(1,len(si2)+1)

print("Calibration Comparison:")
for nm in MODEL_NAMES:
    e,_,_ = ece_calc(results[nm]["probs"].numpy(), results[nm]["labels"])
    print(f"  {DISPLAY[nm]:<22s} ECE={e:.4f}  Top-1={results[nm]['top1']:.2f}%")
print(f"  {'CA-DFN':<22s} ECE={ece_val:.4f}  Top-1={ca_t1:.2f}%")

fig,(a1,a2,a3) = plt.subplots(1,3,figsize=(18,5))
a1.hist(ent[correct],bins=50,alpha=.7,label="Correct",color="#2ecc71",density=True)
a1.hist(ent[~correct],bins=50,alpha=.7,label="Incorrect",color="#e74c3c",density=True)
a1.set_xlabel("Entropy"); a1.set_title("Entropy Distribution",fontweight="bold"); a1.legend()
a2.bar(bcs,bas,width=.05,alpha=.8,color="#3498db",edgecolor="k"); a2.plot([0,1],[0,1],"r--",lw=2)
a2.set_title(f"Reliability (ECE={ece_val:.4f})",fontweight="bold"); a2.set_xlim(0,1); a2.set_ylim(0,1)
covs=np.arange(1,len(si2)+1)/len(si2); i80=int(.8*len(si2))
a3.plot(covs*100,aac*100,color="#2e86de",lw=2)
a3.axhline(ca_t1,color="gray",ls="--",alpha=.5,label=f"Full: {ca_t1:.1f}%")
a3.annotate(f"80%: {aac[i80]*100:.1f}%",xy=(80,aac[i80]*100),xytext=(55,aac[i80]*100-5),
            arrowprops=dict(arrowstyle="->",color="red"))
a3.set_xlabel("Coverage (%)"); a3.set_title("Selective Prediction",fontweight="bold"); a3.legend()
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/uncertainty.png",dpi=300,bbox_inches="tight"); plt.show()
print(f"ECE={ece_val:.4f}  Acc@80%={aac[i80]*100:.2f}%")

## 12. Curriculum Knowledge Distillation (EfficientNet-B4 Student)

In [ ]:
class KDLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.7): super().__init__(); self.T, self.alpha = T, alpha
    def forward(self, s_logits, t_probs, labels):
        s_soft = F.log_softmax(s_logits/self.T, dim=1)
        t_soft = (t_probs+1e-8).pow(1.0/self.T); t_soft = t_soft/t_soft.sum(1,keepdim=True)
        return self.alpha*F.kl_div(s_soft,t_soft,reduction="batchmean")*(self.T**2) + (1-self.alpha)*F.cross_entropy(s_logits,labels)

print("Generating teacher soft labels...")
teacher_probs = []
cadfn.eval()
with torch.no_grad():
    for ps, _ in fus_tr:
        ps = [p.to(CFG.DEVICE) for p in ps]
        ep, _ = cadfn(ps, training=False); teacher_probs.append(ep.cpu())
teacher_probs = torch.cat(teacher_probs)

# Difficulty: ensemble disagreement
print("Computing sample difficulty...")
disagreement = []
for i in range(len(trn_Y)):
    preds = np.stack([trn_P[n][i].numpy() for n in MODEL_NAMES])
    disagreement.append(np.var(preds, axis=0).mean())
disagreement = np.array(disagreement)
difficulty_order = np.argsort(disagreement)

# Student: EfficientNet-B4
student = models.efficientnet_b4(weights="IMAGENET1K_V1")
student.classifier[1] = nn.Linear(student.classifier[1].in_features, CFG.NUM_CLASSES)
student = student.to(CFG.DEVICE)
st_params = sum(p.numel() for p in student.parameters())/1e6
print(f"Student: EfficientNet-B4 ({st_params:.1f}M)")

class _KDDS(Dataset):
    def __init__(self, base, tp, indices=None):
        self.base, self.tp, self.indices = base, tp, indices
    def __len__(self): return len(self.indices) if self.indices is not None else min(len(self.base),len(self.tp))
    def __getitem__(self, i):
        idx = self.indices[i] if self.indices is not None else i
        img,lab = self.base[idx]; return img, lab, self.tp[idx]

kd_crit = KDLoss(CFG.KD_TEMP, CFG.KD_ALPHA)
kd_opt = optim.AdamW([
    {"params":[p for n,p in student.named_parameters() if "classifier" not in n], "lr":1e-4},
    {"params":list(student.classifier.parameters()), "lr":5e-4}], weight_decay=0.01)
kd_sch = CosineAnnealingLR(kd_opt, T_max=CFG.KD_EP, eta_min=1e-6)

best_st = 0.0
print(f"Curriculum KD: {CFG.KD_EP} epochs...")
for ep in range(CFG.KD_EP):
    frac = min(1.0, 0.5 + 0.5*(ep/(CFG.KD_EP*0.7)))
    n_samp = int(frac*len(difficulty_order))
    curr = difficulty_order[:n_samp].tolist()
    kd_ld = DataLoader(_KDDS(train_ds, teacher_probs, curr), CFG.BATCH_SIZE, shuffle=True,
                       num_workers=CFG.WORKERS, pin_memory=True, drop_last=True)
    student.train(); c = t = 0
    for imgs,labs,tp in kd_ld:
        imgs,labs,tp = imgs.to(CFG.DEVICE),labs.to(CFG.DEVICE),tp.to(CFG.DEVICE)
        kd_opt.zero_grad(set_to_none=True); lo = kd_crit(student(imgs),tp,labs)
        lo.backward(); kd_opt.step()
        c += student(imgs).detach().argmax(1).eq(labs).sum().item(); t += labs.size(0)
    kd_sch.step(); vr = evaluate(student, val_ld); flag = ""
    if vr["top1"] > best_st:
        best_st = vr["top1"]; torch.save(student.state_dict(), f"{CFG.CKPT}/student.pth"); flag = " ⭐"
    if (ep+1)%5==0 or flag:
        print(f"  Ep {ep+1:>2}/{CFG.KD_EP} Data {frac*100:.0f}% Val {vr['top1']:.2f}%{flag}")

student.load_state_dict(torch.load(f"{CFG.CKPT}/student.pth", weights_only=True))
st_test = evaluate(student, test_ld)
print(f"\nStudent: Top-1={st_test['top1']:.2f}%  Recovery={st_test['top1']/ca_t1*100:.1f}%  ({st_params:.1f}M params)")

## 13. Efficiency Analysis

In [ ]:
from ptflops import get_model_complexity_info
eff = []; dummy = torch.randn(1,3,224,224).to(CFG.DEVICE)
def _bench(model, label, top1):
    flops,params = get_model_complexity_info(model,(3,224,224),as_strings=False,print_per_layer_stat=False)
    model.eval()
    with torch.no_grad():
        for _ in range(50): model(dummy)
        torch.cuda.synchronize(); t0=time.time()
        for _ in range(200): model(dummy)
        torch.cuda.synchronize(); lat=(time.time()-t0)/200*1000
    eff.append(dict(Model=label, Params_M=params/1e6, GFLOPs=flops/1e9,
                    Latency_ms=lat, Throughput=1000/lat, Top1=top1))

for nm in MODEL_NAMES: _bench(trained[nm], DISPLAY[nm], results[nm]["top1"])
_bench(student, "Student (EffNet-B4 KD)", st_test["top1"])
ens_idx = list(range(len(MODEL_NAMES)))
eff.append(dict(Model="CA-DFN Ensemble",
    Params_M=sum(eff[i]["Params_M"] for i in ens_idx),
    GFLOPs=sum(eff[i]["GFLOPs"] for i in ens_idx),
    Latency_ms=sum(eff[i]["Latency_ms"] for i in ens_idx),
    Throughput=1000/sum(eff[i]["Latency_ms"] for i in ens_idx), Top1=ca_t1))
df_eff = pd.DataFrame(eff)
print(df_eff.to_string(index=False, float_format="%.2f"))
df_eff.to_csv(f"{CFG.OUT}/efficiency.csv", index=False)

fig,ax = plt.subplots(figsize=(10,7))
colors = ["#e74c3c","#3498db","#f39c12","#2ecc71","#9b59b6","#1abc9c","#e67e22"]
for i,row in df_eff.iterrows():
    ax.scatter(row.GFLOPs, row.Top1, s=row.Params_M*3+50, c=colors[i%len(colors)],
              alpha=.8, edgecolors="k", lw=1, zorder=3)
    ax.annotate(row.Model, (row.GFLOPs, row.Top1), xytext=(8,4), textcoords="offset points", fontsize=7)
ax.set_xlabel("GFLOPs"); ax.set_ylabel("Top-1 (%)")
ax.set_title("Accuracy vs Compute (v3 upgraded backbones)", fontweight="bold"); ax.grid(True,alpha=.3)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/pareto.png",dpi=300,bbox_inches="tight"); plt.show()

## 14. Ablation Studies

In [ ]:
print("Ablation 1: Fusion Methods"); print("="*55)
abl = [("Majority Vote",vote_t1),("Uniform Avg",uni["top1"]),
       ("Perf-Weight",prf["top1"]),("CA-DFN (Ours)",ca_t1)]
for m,a in abl: print(f"  {m:<25s} Top-1: {a:.2f}%")

print(f"\nAblation 2: Ensemble Size"); print("="*55)
ranked = sorted(MODEL_NAMES, key=lambda n:results[n]["top1"], reverse=True)
sz_accs = []
for k in range(1,len(ranked)+1):
    r = ens_eval({n:test_P[n] for n in ranked[:k]}, test_Y)
    sz_accs.append(r["top1"]); print(f"  {k} model(s) Top-1={r['top1']:.2f}%")

print(f"\nAblation 3: Leave-One-Out"); print("="*55)
for drop in MODEL_NAMES:
    r = ens_eval({n:test_P[n] for n in MODEL_NAMES if n!=drop}, test_Y)
    print(f"  w/o {DISPLAY[drop]:<18s} Top-1={r['top1']:.2f}%")

mw = ca_w.mean(0).numpy()
fig,(a1,a2,a3) = plt.subplots(1,3,figsize=(18,5))
ms,acs = zip(*abl)
a1.barh(ms,acs,color=["#bdc3c7","#95a5a6","#3498db","#2ecc71"],edgecolor="k")
for i,v in enumerate(acs): a1.text(v+.1,i,f"{v:.2f}%",va="center",fontweight="bold")
a1.set_xlabel("Top-1 (%)"); a1.set_title("Fusion Comparison",fontweight="bold")
a2.plot(range(1,len(ranked)+1),sz_accs,"o-",color="#2e86de",lw=2,ms=8)
a2.set_xlabel("Models"); a2.set_ylabel("Top-1 (%)"); a2.set_title("Ensemble Size",fontweight="bold")
a3.bar([DISPLAY[n] for n in MODEL_NAMES], mw,
       color=["#e74c3c","#3498db","#f39c12","#2ecc71","#9b59b6"],edgecolor="k")
a3.set_ylabel("Weight"); a3.set_title("CA-DFN Weights",fontweight="bold")
a3.tick_params(axis="x",rotation=25)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/ablations.png",dpi=300,bbox_inches="tight"); plt.show()

## 15. Training Curves & Confusion Matrix

In [ ]:
cl = {"resnet50":"#e74c3c","efficientnet_b4":"#3498db","densenet121":"#f39c12",
      "convnext_tiny":"#2ecc71","swin_small":"#9b59b6"}
fig,(a1,a2) = plt.subplots(1,2,figsize=(14,5))
for nm in MODEL_NAMES:
    a1.plot(hists[nm]["ta"],label=DISPLAY[nm],color=cl[nm],lw=1.5)
    a2.plot(hists[nm]["va"],label=DISPLAY[nm],color=cl[nm],lw=1.5)
a1.set_title("Training",fontweight="bold"); a1.legend(fontsize=8)
a2.set_title("Validation",fontweight="bold"); a2.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/training_curves.png",dpi=300,bbox_inches="tight"); plt.show()

cm = confusion_matrix(test_Y,ca_preds); cm_off=cm.copy(); np.fill_diagonal(cm_off,0)
confused=set()
for idx in np.argsort(cm_off.ravel())[-30:]: confused.update(divmod(idx,CFG.NUM_CLASSES))
sel=sorted(confused)[:25]; cm_sub=cm[np.ix_(sel,sel)]
fig,ax=plt.subplots(figsize=(14,12))
sns.heatmap(cm_sub,annot=True,fmt="d",cmap="Blues",ax=ax,
            xticklabels=[CLASS_NAMES[i] for i in sel],yticklabels=[CLASS_NAMES[i] for i in sel])
ax.set_title("Confusion Matrix (Top-25)",fontweight="bold")
plt.xticks(rotation=45,ha="right",fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/confusion.png",dpi=200,bbox_inches="tight"); plt.show()

## 16. CA-DFN Per-Category Weights

In [ ]:
pcw = np.zeros((CFG.NUM_CLASSES, len(MODEL_NAMES)))
for c in range(CFG.NUM_CLASSES):
    m = (test_Y==c)
    if m.sum(): pcw[c] = ca_w[m].mean(0).numpy()
div = pcw.std(1); top_div = np.argsort(div)[-15:]
fig,ax = plt.subplots(figsize=(14,7))
x = np.arange(len(top_div)); w = 0.15
cw = ["#e74c3c","#3498db","#f39c12","#2ecc71","#9b59b6"]
for i,nm in enumerate(MODEL_NAMES):
    ax.bar(x+i*w, pcw[top_div,i], w, label=DISPLAY[nm], color=cw[i], edgecolor="k", lw=.5)
ax.set_xticks(x+w*2); ax.set_xticklabels([CLASS_NAMES[i] for i in top_div],rotation=45,ha="right",fontsize=8)
ax.set_ylabel("Avg Weight"); ax.set_title("CA-DFN Weights per Category",fontweight="bold")
ax.legend(fontsize=8); ax.set_ylim(0,.55)
plt.tight_layout(); plt.savefig(f"{CFG.OUT}/cadfn_weights.png",dpi=300,bbox_inches="tight"); plt.show()

## 17. Final Summary

In [ ]:
print("="*70)
print("  CA-DFN — RESULTS")
print("="*70)
print(f"\nINDIVIDUAL MODELS:")
for nm in MODEL_NAMES:
    r=results[nm]; f1v=f1_score(r["labels"],r["preds"],average="macro")
    print(f"  {DISPLAY[nm]:<20s} Top-1:{r['top1']:.2f}% Top-5:{r['top5']:.2f}% F1:{f1v:.4f}")
print(f"\nENSEMBLE METHODS:")
print(f"  Majority Vote      Top-1: {vote_t1:.2f}%")
print(f"  Uniform Avg        Top-1: {uni['top1']:.2f}%")
print(f"  Perf-Weight        Top-1: {prf['top1']:.2f}%")
print(f"  CA-DFN (Ours)   Top-1: {ca_t1:.2f}%  Top-5: {ca_t5:.2f}%  F1: {ca_f1:.4f}")
print(f"\nKNOWLEDGE DISTILLATION:")
print(f"  Student (EffNet-B4) Top-1: {st_test['top1']:.2f}%  Recovery: {st_test['top1']/ca_t1*100:.1f}%")
print(f"\nANALYSIS:")
print(f"  Complementarity: {mean_comp:.4f}  |  Comp-Gain r={correlation:.3f}")
print(f"  ECE: {ece_val:.4f}  |  Acc@80%: {aac[i80]*100:.2f}%")
print(f"\nFiles saved to {CFG.OUT}/")
for f in sorted(os.listdir(CFG.OUT)): print(f"  {f}")
print("="*70)
print("Pipeline complete. See results/ for all figures and tables.")